# 03 — Previsão da próxima hora

Terceiro passo do fluxo **+1h a partir de agora**. Este notebook carrega o modelo empacotado (`models/modelo_final_sprint5.joblib`, Random Forest) e responde:

> *Daqui a 1 hora, a qualidade do ar em Mogi das Cruzes estará inadequada (IQAr > 100)?*

A decisão usa o limiar congelado do artefato (0,3): probabilidade **≥ 0,3** gera alerta. O limiar é baixo de propósito, para priorizar não perder episódios ruins, ao custo de mais alarmes falsos.

> **Uso experimental.** Os dados são modelados (CAMS/ERA5), não medições locais, e este não é um alerta oficial nem orientação de saúde. O notebook não retreina o modelo e não usa o holdout de 2025–2026.

Pré-requisito: ter executado [`01_Coleta_Dados_Recentes.ipynb`](01_Coleta_Dados_Recentes.ipynb).

## 1. Carregar modelo e dados recentes

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
import yaml

ROOT = Path.cwd()
if not (ROOT / 'config' / 'params.yaml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'config' / 'params.yaml').is_file():
    raise FileNotFoundError('Não encontrei config/params.yaml. Abra o notebook a partir da raiz do projeto.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.aplicacao.Prever_Proxima_Hora import (
    COLUNAS_CONTEXTO_CURTO_PRAZO,
    MODELO_FINAL_PATH,
    _carregar_artefato_modelo,
    _preparar_linha_mais_recente,
)
from src.transformacao.Calcular_Alvo_IQAr import calcular_iqar
from src.transformacao.Features_Temporais import adicionar_resumos_curto_prazo

with (ROOT / 'config' / 'params.yaml').open('r', encoding='utf-8') as arquivo:
    config = yaml.safe_load(arquivo)

caminho_copia = ROOT / config['application']['recent_snapshot']
if not caminho_copia.is_file():
    raise FileNotFoundError(f'Execute antes o notebook 01. Não encontrei: {caminho_copia}')

artefato = _carregar_artefato_modelo(ROOT / MODELO_FINAL_PATH)
pipeline = artefato['pipeline']
features = list(artefato['features'])
limiar = float(artefato['decision_threshold'])
horizonte = pd.Timedelta(hours=int(artefato['target_horizon_hours']))
dados = pd.read_csv(caminho_copia, parse_dates=['time'])

print(f'Modelo: {artefato["model_name"]} | limiar: {limiar} | horizonte: {horizonte}')
print(f'Dados recentes até: {dados["time"].iloc[-1]}')

Modelo: random_forest_balanced | limiar: 0.3 | horizonte: 0 days 01:00:00
Dados recentes até: 2026-10-20 23:00:00


## 2. Prever a próxima hora

In [2]:
entrada = _preparar_linha_mais_recente(dados, features)
if not hasattr(pipeline, 'predict_proba'):
    raise TypeError('O pipeline empacotado não fornece predict_proba.')

probabilidade = float(pipeline.predict_proba(entrada[features])[0, 1])
alerta = probabilidade >= limiar
horario = entrada['time'].iloc[0]

resumo = pd.DataFrame([{
    'horário dos dados até t': horario,
    'horário do evento previsto': horario + horizonte,
    'probabilidade estimada de IQAr > 100': probabilidade,
    'limiar congelado': limiar,
    'resultado da regra': 'ALERTA (regra experimental)' if alerta else 'Sem alerta pela regra',
}])
display(resumo)
print('Previsão operacional +1h com dados recentes. Apoio experimental, não é alerta oficial.')

,horário dos dados até t,horário do evento previsto,probabilidade estimada de IQAr > 100,limiar congelado,resultado da regra
0,2026-10-20 23:00:00,2026-10-21,0.0,0.3,Sem alerta pela regra


Previsão operacional +1h com dados recentes. Apoio experimental, não é alerta oficial.


## 3. Contexto: as últimas horas

Para ajudar a interpretar o resultado, a tabela abaixo mostra a probabilidade do modelo e o IQAr em cada uma das últimas horas com features completas. Quando a hora seguinte também já está disponível, aparece o IQAr que de fato ocorreu (`iqar na hora seguinte`), permitindo uma conferência rápida e informal — não é uma métrica de desempenho.

In [3]:
base = adicionar_resumos_curto_prazo(calcular_iqar(dados), COLUNAS_CONTEXTO_CURTO_PRAZO)
completas = base.dropna(subset=features).copy()
completas['probabilidade'] = pipeline.predict_proba(completas[features])[:, 1]
completas['alerta'] = completas['probabilidade'] >= limiar
completas['iqar na hora seguinte'] = completas['iqar'].shift(-1)

completas[['time', 'iqar', 'probabilidade', 'alerta', 'iqar na hora seguinte']].tail(12).reset_index(drop=True)

,time,iqar,probabilidade,alerta,iqar na hora seguinte
0,2026-10-20 12:00:00,21.80,0.0,False,23.90
1,2026-10-20 13:00:00,23.90,0.0,False,26.15
2,2026-10-20 14:00:00,26.15,0.0,False,27.95
3,2026-10-20 15:00:00,27.95,0.0,False,28.80
4,2026-10-20 16:00:00,28.80,0.0,False,28.75
5,2026-10-20 17:00:00,28.75,0.0,False,28.15
6,2026-10-20 18:00:00,28.15,0.0,False,26.95
7,2026-10-20 19:00:00,26.95,0.0,False,25.25
8,2026-10-20 20:00:00,25.25,0.0,False,23.20
9,2026-10-20 21:00:00,23.20,0.0,False,21.10


## Como repetir

Para uma nova previsão, execute novamente o notebook 01 (atualiza os dados) e este notebook. Sem notebook, basta:

```powershell
.\.venv\Scripts\python.exe -m src.aplicacao.Prever_Proxima_Hora
```